# Lecture : Graph Convolutional Networks

## Lab 05 : GatedGCNs for chemical regression -- Solution -- PyTorch Geometric version

### Xavier Bresson, Nian Liu, Ryoji Kubo

Bresson, Laurent, Residual Gated Graph ConvNets, 2017  
https://arxiv.org/pdf/1711.07553

Bresson, Laurent, A two-step graph convolutional decoder for molecule generation, 2019  
https://arxiv.org/pdf/1906.03412

This notebook is the [PyTorch Geometric (PyG)](https://pytorch-geometric.readthedocs.io) counterpart of
`code05_solution.ipynb`. The molecular dataset is still the one shipped with the course (DGL graphs saved on disk),
but every molecule is converted into a PyG `Data` object and the **GatedGCN is implemented from scratch** with PyG's
`MessagePassing` class (we do *not* call a ready-made convolution such as `torch_geometric.nn.ResGatedGraphConv`).

In [1]:
# For Google Colaboratory
import sys, os
if 'google.colab' in sys.modules:
    # mount google drive
    from google.colab import drive
    drive.mount('/content/gdrive')
    path_to_file = '/content/gdrive/My Drive/CS5284_2026_codes/06_Graph_Convnets'
    print(path_to_file)
    # change current path to the folder containing "path_to_file"
    os.chdir(path_to_file)
    !pwd
    !pip -q install torch_geometric                                            # Install PyG
    !pip -q install -f https://data.dgl.ai/wheels/torch-2.6/repo.html dgl==2.5.0 # Install DGL (dataset only)


In [2]:
# Libraries
import pickle
from lib.utils import Dictionary, MoleculeDataset, MoleculeDGL, Molecule
from torch.utils.data import DataLoader
import dgl                                  # only used to load the molecular dataset
import torch
from torch import Tensor
import torch.nn as nn
import time

# PyTorch Geometric
import torch_geometric
from torch_geometric.data import Data, Batch
from torch_geometric.nn import MessagePassing, global_mean_pool

print('torch:', torch.__version__)
print('torch_geometric:', torch_geometric.__version__)
print('dgl:', dgl.__version__)

torch: 2.1.2
torch_geometric: 2.5.1
dgl: 1.0.2


# Load molecular datasets

The molecules are stored on disk as DGL graphs. A PyG graph is a `Data` object holding plain tensors:
  - `data.edge_index` of size `(2, num_edges)`: `edge_index[0]` = source nodes $j$, `edge_index[1]` = destination nodes $i$
    (PyG's default message flow is `source_to_target`, i.e. $j \rightarrow i$),
  - `data.x` of size `(num_nodes,)`: the atom types (integers, fed to an `nn.Embedding`),
  - `data.edge_attr` of size `(num_edges,)`: the bond types (integers, fed to an `nn.Embedding`),
  - `data.y`: the chemical property to regress, one scalar per molecule.

so the conversion simply moves `g.edges()`, `g.ndata['feat']` and `g.edata['feat']` into these fields.

In [3]:
# Select dataset

print('Loading data')
data_folder_pytorch = 'datasets/QM9_pytorch/'
with open(data_folder_pytorch+"train_pytorch.pkl","rb") as f:
    dataset=pickle.load(f)

# Load the number of atom and bond types 
with open(data_folder_pytorch + "atom_dict.pkl" ,"rb") as f: num_atom_type = len(pickle.load(f))
with open(data_folder_pytorch + "bond_dict.pkl" ,"rb") as f: num_bond_type = len(pickle.load(f))
print(num_atom_type)
print(num_bond_type)

# Load the DGL datasets
data_folder_dgl = 'datasets/QM9_dgl/'
dataset_name = 'QM9'
datasets_dgl = MoleculeDataset(dataset_name, data_folder_dgl)


# Convert a DGL molecule into a PyG Data object
def dgl_to_pyg(dgl_graph, label):
    src, dst = dgl_graph.edges()                                   # src = j (source), dst = i (destination)
    edge_index = torch.stack([src.long(), dst.long()], dim=0)      # (2,num_edges)
    x = dgl_graph.ndata['feat'].long()                             # (num_nodes,)   atom type
    edge_attr = dgl_graph.edata['feat'].long()                     # (num_edges,)   bond type
    y = label.view(1, -1)                                          # (1,1)          chemical target
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, num_nodes=dgl_graph.num_nodes())

# Convert a whole DGL dataset into a list of PyG Data objects
def convert_dataset(dgl_dataset):
    return [ dgl_to_pyg(graph, label) for (graph, label) in dgl_dataset ]

trainset = convert_dataset(datasets_dgl.train)
valset   = convert_dataset(datasets_dgl.val)
testset  = convert_dataset(datasets_dgl.test)
print(len(trainset))
print(len(valset))
print(len(testset))
idx = 0
print(trainset[:2])
print(valset[idx])
print(testset[idx])

Loading data
13
4
Loading datasets QM9_dgl...
train, test, val sizes : 2000 200 200
Time: 0.6246s
2000
200
200
[Data(x=[9], edge_index=[2, 20], edge_attr=[20], y=[1, 1], num_nodes=9), Data(x=[9], edge_index=[2, 18], edge_attr=[18], y=[1, 1], num_nodes=9)]
Data(x=[9], edge_index=[2, 18], edge_attr=[18], y=[1, 1], num_nodes=9)
Data(x=[9], edge_index=[2, 20], edge_attr=[20], y=[1, 1], num_nodes=9)


# Generate a batch of graphs and test it

As in DGL, batching builds a **block-diagonal** adjacency matrix: the node and edge features are concatenated and
the node indices stored in `edge_index` are shifted by the cumulated number of nodes of the previous graphs. The
batched graph additionally carries a vector `batch` of size `(num_nodes_of_the_batch,)` that maps every node to the
index of the graph it belongs to -- this is what the graph-level readout will use.

In [4]:
# collate function prepares a batch of graphs, labels and other graph features
def collate(samples):
    
    # Input sample is a list of PyG Data objects (each one carries its own label in .y)
    batch_graphs = Batch.from_data_list(samples) # batch of graphs
    batch_labels = batch_graphs.y                # batch of labels (here chemical target)
    
    # Normalization w.r.t. graph sizes
    tab_sizes_n = [ samples[i].num_nodes for i in range(len(samples))]
    tab_norm_n = [ torch.FloatTensor(size,1).fill_(1./float(size)) for size in tab_sizes_n ]
    batch_norm_n = torch.cat(tab_norm_n).sqrt()  
    tab_sizes_e = [ samples[i].num_edges for i in range(len(samples))]
    tab_norm_e = [ torch.FloatTensor(size,1).fill_(1./float(size)) for size in tab_sizes_e ]
    batch_norm_e = torch.cat(tab_norm_e).sqrt()
    
    return batch_graphs, batch_labels, batch_norm_n, batch_norm_e


# Generate a batch of graphs
batch_size = 10
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True, collate_fn=collate)
batch_graphs, batch_labels, batch_norm_n, batch_norm_e = list(train_loader)[0]
print(batch_graphs)
print(batch_labels)
print('batch_norm_n:',batch_norm_n.size())
print('batch_norm_e:',batch_norm_e.size())
batch_x = batch_graphs.x
print('batch_x:',batch_x.size())
batch_e = batch_graphs.edge_attr
print('batch_e:',batch_e.size())
print('batch vector (node => graph index):',batch_graphs.batch.size())

DataBatch(x=[90], edge_index=[2, 204], edge_attr=[204], y=[10, 1], num_nodes=90, batch=[90], ptr=[11])
tensor([[-3.3593],
        [ 2.4529],
        [ 1.5436],
        [-0.5803],
        [-0.1021],
        [ 0.8400],
        [-0.2995],
        [ 2.5237],
        [ 0.4236],
        [ 0.3152]])
batch_norm_n: torch.Size([90, 1])
batch_norm_e: torch.Size([204, 1])
batch_x: torch.Size([90])
batch_e: torch.Size([204])
batch vector (node => graph index): torch.Size([90])


## Question 1: Implement a two-layer MLP for regression on a top of a GatedGCN network

Node and edge update equations for GatedGCN layers:  
\begin{align}
h_i^{\ell+1} &=& h_i^{\ell} + \text{ReLU} \Big( \text{BN} \Big( A^\ell h_i^{\ell} +  \sum_{j\sim i} \eta(e_{ij}^{\ell}) \odot B^\ell h_j^{\ell} \Big) \Big), \quad \eta(e_{ij}^{\ell}) = \frac{\sigma(e_{ij}^{\ell})}{\sum_{j'\sim i} \sigma(e_{ij'}^{\ell}) + \varepsilon} \\
e_{ij}^{\ell+1} &=& e^\ell_{ij} + \text{ReLU} \Big( \text{BN}  \Big( C^\ell e_{ij}^{\ell} + D^\ell h^{\ell}_i + E^\ell h^{\ell}_j  \Big) \Big)
\end{align}

MLP block for scalar regression of a molecular property:
\begin{align}
y &=& \textrm{MLP}(\hat{h})\in\mathbb{R} \\
\hat{h} &=& \frac{1}{n} \sum_{i=1}^n h_i^{\ell=L}\in\mathbb{R}^d\\
\textrm{MLP}(h) &=& \textrm{LL}_2\big( \textrm{ReLU} \big( \textrm{LL}_1 \big(h\big) \big) \big), h\in\mathbb{R}^d
\end{align}

PyG implements message-passing with the class
[`MessagePassing`](https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html#torch_geometric.nn.MessagePassing),
whose two entry points are the counterparts of DGL's `message_func()` / `reduce_func()`:

  - `self.edge_updater(edge_index, ...)` calls `edge_update()` and returns **one feature per edge**. This is where the
    new edge feature $e_{ij}^{\ell+1}$ is computed -- in DGL this was done as a side effect inside `message_func()`
    with `edges.data['e'] = eij`.
  - `self.propagate(edge_index, ...)` calls `message()` and then aggregates the messages on the destination nodes $i$
    with the constructor argument `aggr='add'`, giving $\sum_{j\sim i}$.

Any tensor `xx` given to `propagate()`/`edge_updater()` is read inside `message()`/`edge_update()` as `xx_j` (feature
of the **source** node $j$, i.e. DGL's `edges.src['xx']`), as `xx_i` (feature of the **destination** node $i$, i.e.
`edges.dst['xx']`), or as `xx` when it already lives on the edges (i.e. `edges.data['xx']`). PyG performs the lifting
automatically: `xx_j` is nothing but `xx[edge_index[0]]`.

Note on `propagate_type` / `edge_updater_type`: PyG compiles a specialized `propagate()` and `edge_updater()` for every
layer class, and it usually infers the names/types of their arguments by *parsing the source code* of the class. This
parsing is unavailable for a class defined in a notebook cell, so the two class attributes declare them explicitly.
Keep them whenever you write a `MessagePassing` layer in a notebook.

In [5]:
# Define a two-layer MLP for regression 
class MLP_layer(nn.Module): 
    
    def __init__(self, input_dim, hidden_dim): 
        super(MLP_layer, self).__init__()
        ########################################
        # YOUR CODE STARTS
        # Implement a two-layer predictor
        ########################################
        self.linear1 = nn.Linear( input_dim, hidden_dim, bias=True )
        self.linear2 = nn.Linear( hidden_dim, 1, bias=True )   
        ########################################
        # YOUR CODE ENDS
        ########################################
        
    def forward(self, x):
        ########################################
        # YOUR CODE STARTS
        # Pass `x` through the two-layer MLP and get prediction y
        ########################################
        y = self.linear2(torch.relu(self.linear1(x)))
        ########################################
        # YOUR CODE ENDS
        ########################################
        return y


# class of GatedGCN layer  
class GatedGCN_layer(MessagePassing):
    
    # Declare the type of the tensors given to edge_updater() and propagate(), see the note above
    edge_updater_type = {'Ce': Tensor, 'Dh': Tensor, 'Eh': Tensor}
    propagate_type = {'Bh': Tensor, 'e': Tensor}
    
    def __init__(self, input_dim, output_dim):
        # aggr='add' : the messages arriving at node i are summed  => sum_{j~i}
        # flow='source_to_target' : edge_index[0]=j is the source, edge_index[1]=i is the destination
        super(GatedGCN_layer, self).__init__(aggr='add', flow='source_to_target')
        self.A = nn.Linear(input_dim, output_dim, bias=True)
        self.B = nn.Linear(input_dim, output_dim, bias=True)
        self.C = nn.Linear(input_dim, output_dim, bias=True)
        self.D = nn.Linear(input_dim, output_dim, bias=True)
        self.E = nn.Linear(input_dim, output_dim, bias=True)
        self.bn_node_h = nn.BatchNorm1d(output_dim)
        self.bn_node_e = nn.BatchNorm1d(output_dim)

    # Step 1 of message-passing with PyG: 
    #   edge_update() computes one feature per edge (src/j => dst/i), called by self.edge_updater()
    def edge_update(self, Ce, Dh_i, Eh_j):
        eij = Ce + Dh_i + Eh_j # Ceij + Dhi + Ehj with dst/i, src/j
        return eij # return the updated edge feature value

    # Step 2 of message-passing with PyG: 
    #   message() builds the message sent by node src/j to node dst/i, and propagate() sums them over j~i.
    #   The gate eta_ij = sigma_ij / sum_j' sigma_ij' needs two sums over the neighbours: the numerator
    #   sum_j sigma_ij * Bhj and the denominator sum_j sigma_ij. DGL could compute them separately because
    #   reduce_func() materializes a mailbox; PyG aggregates with a single scatter operation, so we stack
    #   the two quantities along the feature dimension and split them again after propagate().
    def message(self, Bh_j, e):
        sigmaij = torch.sigmoid(e) # sigma_ij = sigmoid(e_ij)
        return torch.cat([sigmaij * Bh_j, sigmaij], dim=-1) # send message={sigma_ij*Bhj, sigma_ij} to node dst/i
    
    def forward(self, h, e, edge_index, snorm_n, snorm_e):
        
        h_in = h # residual connection
        e_in = e # residual connection
        
        Ah = self.A(h) # linear transformation 
        Bh = self.B(h) # linear transformation 
        Dh = self.D(h) # linear transformation 
        Eh = self.E(h) # linear transformation 
        Ce = self.C(e) # linear transformation 
        
        e = self.edge_updater(edge_index, Ce=Ce, Dh=Dh, Eh=Eh) # update the edge features with PyG
        
        out = self.propagate(edge_index, Bh=Bh, e=e, size=(h.size(0), h.size(0))) # update the node features with PyG
        sum_sigma_Bh, sum_sigma = out.chunk(2, dim=-1) # split the numerator and the denominator of the gate
        h = Ah + sum_sigma_Bh / sum_sigma # hi = Ahi + sum_j eta_ij * Bhj
        
        h = h* snorm_n # normalize activation w.r.t. graph node size
        e = e* snorm_e # normalize activation w.r.t. graph edge size
        
        h = self.bn_node_h(h) # batch normalization  
        e = self.bn_node_e(e) # batch normalization  
        
        h = torch.relu(h) # non-linear activation
        e = torch.relu(e) # non-linear activation
        
        h = h_in + h # residual connection
        e = e_in + e # residual connection
        
        return h, e

### What do `edge_updater()` and `propagate()` actually do?

They are *not* black boxes: they lift the node tensors to the edges, call `edge_update()` / `message()`, and scatter
the result back to the destination nodes. The cell below re-implements one GatedGCN layer with plain PyTorch indexing
(`Bh[src]` for the lifting, `index_add_` for the sum over the neighbours) and checks that it returns exactly the same
node and edge features.

In [6]:
# One GatedGCN layer written without MessagePassing, to show what edge_updater()/propagate() compute
def gatedgcn_layer_from_scratch(layer, h, e, edge_index, snorm_n, snorm_e):
    src, dst = edge_index[0], edge_index[1]                 # src = j, dst = i
    h_in, e_in = h, e
    Ah, Bh, Dh, Eh, Ce = layer.A(h), layer.B(h), layer.D(h), layer.E(h), layer.C(e)
    
    e = Ce + Dh[dst] + Eh[src]                              # edge_update() : "_i"/"_j" lifting = indexing with dst/src
    sigmaij = torch.sigmoid(e)                              # message()
    num = torch.zeros_like(Ah).index_add_(0, dst, sigmaij * Bh[src])  # aggregate(aggr='add') : sum_{j~i}
    den = torch.zeros_like(Ah).index_add_(0, dst, sigmaij)
    h = Ah + num / den
    
    h, e = h * snorm_n, e * snorm_e
    h, e = layer.bn_node_h(h), layer.bn_node_e(e)
    h, e = torch.relu(h), torch.relu(e)
    return h_in + h, e_in + e

with torch.no_grad():
    D = 64
    layer = GatedGCN_layer(D, D)
    h = torch.randn(batch_graphs.num_nodes, D)
    e = torch.randn(batch_graphs.num_edges, D)
    h_pyg, e_pyg = layer(h, e, batch_graphs.edge_index, batch_norm_n, batch_norm_e)
    h_scratch, e_scratch = gatedgcn_layer_from_scratch(layer, h, e, batch_graphs.edge_index, batch_norm_n, batch_norm_e)
    print('max |h_pyg - h_from_scratch| =', (h_pyg - h_scratch).abs().max().item())
    print('max |e_pyg - e_from_scratch| =', (e_pyg - e_scratch).abs().max().item())

max |h_pyg - h_from_scratch| = 0.0
max |e_pyg - e_from_scratch| = 0.0


## Question 2: Implement the Mean Absolute Error (MAE) loss

Hint: You may use [torch.nn.L1Loss()](https://pytorch.org/docs/stable/generated/torch.nn.L1Loss.html#torch.nn.L1Loss).
                                       

In [7]:
class GatedGCN_net(nn.Module):
    
    def __init__(self, net_parameters):
        super(GatedGCN_net, self).__init__()
        input_dim = net_parameters['input_dim']
        hidden_dim = net_parameters['hidden_dim']
        L = net_parameters['L']
        self.embedding_h = nn.Embedding(num_atom_type, hidden_dim)
        self.embedding_e = nn.Embedding(num_bond_type, hidden_dim)
        self.GatedGCN_layers = nn.ModuleList([ GatedGCN_layer(hidden_dim, hidden_dim) for _ in range(L) ]) 
        self.MLP_layer = MLP_layer(hidden_dim, hidden_dim)
        
    def forward(self, h, e, edge_index, batch_vector, snorm_n, snorm_e):
        
        # input embedding
        h = self.embedding_h(h)
        e = self.embedding_e(e)
        
        # graph convnet layers
        for GGCN_layer in self.GatedGCN_layers:
            h,e = GGCN_layer(h,e,edge_index,snorm_n,snorm_e)
        
        # MLP classifier
        # global_mean_pool() averages the node features of each graph of the batch, using the "batch" vector
        # that maps every node to its graph index. It is the PyG counterpart of dgl.mean_nodes().
        y = global_mean_pool(h, batch_vector) # PyG mean function over the nodes of each graph
        y = self.MLP_layer(y)
        
        return y    
    
    def loss(self, y_scores, y_labels):
        ########################################
        # YOUR CODE START
        # Define the Mean Absolute Error (MAE) as regression loss 
        ########################################
        loss = nn.L1Loss()(y_scores, y_labels)
        ########################################
        # YOUR CODE END
        ########################################
        return loss        
    
    def update(self, lr):       
        update = torch.optim.Adam( self.parameters(), lr=lr )
        return update


# Instantiate one network (testing)
net_parameters = {}
net_parameters['input_dim'] = 1
net_parameters['hidden_dim'] = 128
net_parameters['L'] = 4
net = GatedGCN_net(net_parameters)
print(net)

GatedGCN_net(
  (embedding_h): Embedding(13, 128)
  (embedding_e): Embedding(4, 128)
  (GatedGCN_layers): ModuleList(
    (0-3): 4 x GatedGCN_layer()
  )
  (MLP_layer): MLP_layer(
    (linear1): Linear(in_features=128, out_features=128, bias=True)
    (linear2): Linear(in_features=128, out_features=1, bias=True)
  )
)


# Train the network

In [8]:
def run_one_epoch(net, data_loader, train=True):
    if train:
        net.train() # during training
    else:
        net.eval()  # during inference/test
    epoch_loss = 0
    nb_data = 0
    gpu_mem = 0
    for iter, (batch_graphs, batch_labels, batch_snorm_n, batch_snorm_e) in enumerate(data_loader):
        batch_x = batch_graphs.x
        batch_e = batch_graphs.edge_attr
        batch_edge_index = batch_graphs.edge_index
        batch_vector = batch_graphs.batch
        batch_snorm_n = batch_snorm_n
        batch_snorm_e = batch_snorm_e
        batch_labels = batch_labels
        batch_scores = net.forward(batch_x, batch_e, batch_edge_index, batch_vector, batch_snorm_n, batch_snorm_e)
        loss = net.loss(batch_scores, batch_labels)
        if train: # during training, run backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        epoch_loss += loss.detach().item()
        nb_data += batch_labels.size(0)
    epoch_loss /= (iter + 1)
    return epoch_loss


# dataset loaders
# (the original notebook used datasets_dgl.collate here, which is the same function as the collate() above)
batch_size = 100
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True, collate_fn=collate)
test_loader = DataLoader(testset, batch_size=batch_size, shuffle=True, collate_fn=collate)
val_loader = DataLoader(valset, batch_size=batch_size, shuffle=True, collate_fn=collate)

# Instantiate one network
net_parameters = {}
net_parameters['input_dim'] = 1
net_parameters['hidden_dim'] = 100
net_parameters['L'] = 4
net = GatedGCN_net(net_parameters)

# optimizer
optimizer = torch.optim.Adam(net.parameters(), lr=0.0001)

# training loop
for epoch in range(50):
    start = time.time()
    epoch_train_loss = run_one_epoch(net, train_loader, True)
    with torch.no_grad(): 
        epoch_test_loss = run_one_epoch(net, test_loader, False)
        epoch_val_loss = run_one_epoch(net, val_loader, False)  
    print('Epoch {}, time {:.4f}, train_loss: {:.4f}, test_loss: {:.4f}, val_loss: {:.4f}'.format(epoch, time.time()-start, epoch_train_loss, epoch_test_loss, epoch_val_loss))

Epoch 0, time 0.4491, train_loss: 1.2992, test_loss: 1.3010, val_loss: 1.1932
Epoch 1, time 0.4305, train_loss: 1.2006, test_loss: 1.2015, val_loss: 1.0801
Epoch 2, time 0.4993, train_loss: 1.1433, test_loss: 1.1016, val_loss: 1.0011
Epoch 3, time 0.4784, train_loss: 1.0979, test_loss: 1.0407, val_loss: 0.9596
Epoch 4, time 0.4784, train_loss: 1.0571, test_loss: 0.9896, val_loss: 0.9136
Epoch 5, time 0.4485, train_loss: 1.0142, test_loss: 0.9433, val_loss: 0.8741
Epoch 6, time 0.4446, train_loss: 0.9749, test_loss: 0.9114, val_loss: 0.8302
Epoch 7, time 0.4565, train_loss: 0.9326, test_loss: 0.8880, val_loss: 0.7819
Epoch 8, time 0.4462, train_loss: 0.8990, test_loss: 0.8356, val_loss: 0.7627
Epoch 9, time 0.4369, train_loss: 0.8660, test_loss: 0.8547, val_loss: 0.7707
Epoch 10, time 0.4371, train_loss: 0.8323, test_loss: 0.7887, val_loss: 0.6972
Epoch 11, time 0.4334, train_loss: 0.8039, test_loss: 0.7497, val_loss: 0.6689
Epoch 12, time 0.4400, train_loss: 0.7864, test_loss: 0.7399, 